## 1 - Loading Processed Data

In this step, we load processed video dataset from the previously notebooks


In [7]:
import pandas as pd
import os

In [8]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fplusplus_extracted_frames.csv"
fplusplus_imgs = pd.read_csv(load_path)

print(f"{len(fplusplus_imgs)} loaded!")
pd.set_option('display.max_colwidth', None)
display(fplusplus_imgs.sample(5))

load_path = "./processed_images/fei_images_final.csv"
fei_imgs = pd.read_csv(load_path)

print(f"{len(fei_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(fei_imgs.sample(5))

load_path = "./processed_images/celeb_frames.csv"
celeb_imgs = pd.read_csv(load_path)

print(f"{len(celeb_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_imgs.sample(5))

--- LOADING PROCESSED DATA ---
70000 loaded!


,path,label,split,dataset,method
32041,F++_Split_Intra/target_based/val/fake/FaceSwap_981_985_f2_ssd.jpg,1,val,FF++,target_based
62973,F++_Split_Intra/source_based/train/fake/FaceShifter_252_266_f3_ssd.jpg,1,train,FF++,source_based
63156,F++_Split_Intra/source_based/train/fake/FaceShifter_238_282_f0_ssd.jpg,1,train,FF++,source_based
68414,F++_Split_Intra/source_based/val/fake/NeuralTextures_192_134_f2_ssd.jpg,1,val,FF++,source_based
13962,F++_Split_Intra/target_based/train/fake/DeepFakeDetection_06_18__walking_down_indoor_hall_disgust__DEA1TCLN_f4_ssd.jpg,1,train,FF++,target_based


8054 images loaded!


,path,label,split,dataset,method
6828,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_122-11_177-11_C08_B50_W50_PA08_PM00_F00_ssd.png,1,train,FEI,NaN
7342,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_181-11_168-11_C03_B30_W30_PA03_PM00_F00_ssd.png,1,train,FEI,NaN
2413,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_113-11_20-11_C15_B30_W30_PA15_PM00_F00_ssd.png,1,train,FEI,NaN
1437,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_116-11_4-11_C02_B30_W30_PA02_PM00_F00_ssd.png,1,train,FEI,NaN
3053,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_79-11_132-11_C03_B50_W50_PA03_PM00_F00_ssd.png,1,train,FEI,NaN


1780 images loaded!


,path,label,split,dataset,method
620,CelebDF_Test/original/00011_00011_original_00011_f0_ssd.jpg,0,test,CelebDF,NaN
1564,CelebDF_Test/fake/id47_id39_TPSMM_id39_id47_0000_f0_ssd.jpg,1,test,CelebDF,NaN
987,CelebDF_Test/original/00156_00156_original_00156_f0_ssd.jpg,0,test,CelebDF,NaN
436,CelebDF_Test/fake/id0_id17_TPSMM_id17_id0_0004_f0_ssd.jpg,1,test,CelebDF,NaN
1736,CelebDF_Test/fake/id35_id20_HyperReenact_id20_id35_0009_f0_ssd.jpg,1,test,CelebDF,NaN


## 2 - Master Dataset Compilation & Data Export

In this final preprocessing step, we consolidate our distinct datasets into standardized structures:

1. **Format Standardization:** We unify the column structure (`path`, `label`, `split`, `dataset`, `method`) across all dataframes and explicitly cast the labels into standard integers (`0` for Real, `1` for Fake).
2. **Master Dataset Assembly (FEI + FF++):** We concatenate the FEI and FaceForensics++ dataframes into a single, shuffled dataset. This `master_df` is exported as `master_dataset.csv` and contains our primary **Train**, **Validation**, and **Internal Test** splits.
3. **External Test Isolation (Celeb-DF):** The Celeb-DF dataset is deliberately excluded from the master dataset. It is held out completely as an **External Test Set**, serving exclusively for evaluating the model's cross-dataset generalization capabilities.
4. **Final Audit:** We output grouped distribution tables and random samples to verify the final dataset composition, split proportions, and label balancing.

In [ ]:
df_fei = fei_imgs[['path', 'label', 'split']].copy()
df_fei['dataset'] = 'FEI'
df_fei['method'] = 'N/A' 
df_fei['label'] = df_fei['label'].replace({'original': 0, 'fake': 1}).astype(int) 

df_ff = fplusplus_imgs[['path', 'label', 'split', 'method']].copy()
df_ff['dataset'] = 'FF++'
df_ff['label'] = df_ff['label'].astype(int)

df_celeb = celeb_imgs[['path', 'label', 'split']].copy()
df_celeb['label'] = df_celeb['label'].astype(int)

master_df = pd.concat([df_fei, df_ff], ignore_index=True)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
master_df.to_csv("master_dataset.csv", index=False)
print("Merge completed! File saved as 'master_dataset.csv'.")

print("--- MASTER DATASET DISTRIBUTION (FEI + FF++) ---")
distribution_master = master_df.groupby(['dataset', 'method', 'split', 'label']).size().unstack(fill_value=0)
if len(distribution_master.columns) == 2:
    distribution_master.columns = ['0 (Real)', '1 (Fake)']
display(distribution_master)

print("\n--- CELEB-DF DISTRIBUTION (EXTERNAL TEST SET) ---")
distribution_celeb = df_celeb.groupby(['path', 'split', 'label']).size().unstack(fill_value=0)
if len(distribution_celeb.columns) == 2:
    distribution_celeb.columns = ['0 (Real)', '1 (Fake)']
display(distribution_celeb)


print("\n--- SAMPLES ---")
print("Master Dataset Sample (Train/Val/Test interni):")
display(master_df.sample(5))

print("\nCeleb-DF Sample (Test esterno):")
display(df_celeb.sample(5))

Merge completed! File saved as 'master_dataset.csv'.
--- MASTER DATASET DISTRIBUTION (FEI + FF++) ---


0 (Real)  1 (Fake)
dataset method       split                    
FEI     N/A          test         30       322
                     train       140      7322
                     val          30       210
FF++    source_based test        750      4645
                     train      3500     20810
                     val         750      4545
        target_based test        750      4735
                     train      3500     20600
                     val         750      4665


--- CELEB-DF DISTRIBUTION (EXTERNAL TEST SET) ---


,,0 (Real),1 (Fake)
path,split,,
CelebDF_Test/fake/id00061_id54_Real3DPortrait_id54_0009_test_id00061_nrI2uwhFFto_f0_ssd.jpg,test,0,1
CelebDF_Test/fake/id00081_id22_AniTalker_id22_0003_test_id00081_jO5DSKhhcqQ_f0_ssd.jpg,test,0,1
CelebDF_Test/fake/id00081_id22_FLOAT_id22_0003_test_id00081_jO5DSKhhcqQ_f0_ssd.jpg,test,0,1
CelebDF_Test/fake/id00081_id22_SadTalker_id22_0003_test_id00081_jO5DSKhhcqQ_f0_ssd.jpg,test,0,1
CelebDF_Test/fake/id00081_id27_Real3DPortrait_id27_0003_test_id00081_S4aLSTDVi44_f0_ssd.jpg,test,0,1
...,...,...,...
CelebDF_Test/original/id9_id9_original_id9_0005_f0_ssd.jpg,test,1,0
CelebDF_Test/original/id9_id9_original_id9_0006_f0_ssd.jpg,test,1,0
CelebDF_Test/original/id9_id9_original_id9_0007_f0_ssd.jpg,test,1,0



--- SAMPLES ---
Master Dataset Sample (Train/Val/Test interni):


,path,label,split,dataset,method
63200,F++_Split_Intra/source_based/val/fake/FaceShifter_686_696_f0_ssd.jpg,1,val,FF++,source_based
61135,F++_Split_Intra/target_based/train/original/original_173_f3_ssd.jpg,0,train,FF++,target_based
69009,F++_Split_Intra/source_based/val/fake/FaceSwap_194_235_f2_ssd.jpg,1,val,FF++,source_based
14711,F++_Split_Intra/target_based/train/fake/FaceSwap_469_481_f2_ssd.jpg,1,train,FF++,target_based
21125,F++_Split_Intra/target_based/train/fake/DeepFakeDetection_24_19__walk_down_hall_angry__OF8LF68K_f1_ssd.jpg,1,train,FF++,target_based



Celeb-DF Sample (Test esterno):


,path,label,split
57,CelebDF_Test/fake/id17_id26_Celeb-DF-v2_id26_id17_0008_f0_ssd.jpg,1,test
1729,CelebDF_Test/original/id7_id7_original_id7_0002_f0_ssd.jpg,0,test
23,CelebDF_Test/original/id26_id26_original_id26_0003_f0_ssd.jpg,0,test
1375,CelebDF_Test/original/00194_00194_original_00194_f0_ssd.jpg,0,test
1024,CelebDF_Test/original/id1_id1_original_id1_0009_f0_ssd.jpg,0,test


## 3 - Saving Data & Exporting Archive

To conclude this notebook, we first save our fully cleaned and processed DataFrames to local CSV files (e.g., `master_dataset.csv`, `celeb_test_dataset.csv`). This ensures our prepared metadata is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the intensive preprocessing and extraction steps.

Finally, we package the entire structured dataset into a single, highly portable archive (`deepfake_dataset.zip`). This makes it easy to download, store, or transfer the data for model training.

**Contents of the final archive:**
* `F++_Split_Intra/`: The processed and split FaceForensics++ frames.
* `FEI_Split_Intra/`: The processed and split FEI dataset frames.
* `CelebDF_Test/`: The completely held-out Celeb-DF frames for cross-dataset testing.
* `master_dataset.csv`: The unified metadata for the training, validation, and internal test sets.
* `celeb_test_dataset.csv`: The metadata specifically for the external Celeb-DF test set.

*(Note: We use the `-r` flag to include all subdirectories recursively and the `-q` flag to run the compression quietly, keeping the notebook output clean).*

In [12]:
print("--- SAVING DATAFRAME ---")

save_path_master = "master_dataset.csv"
save_path_test = "celeb_test_dataset.csv"

master_df.to_csv(save_path_master, index=False)
df_celeb.to_csv(save_path_test, index=False)

print(f"Master Data successfully saved to: {save_path_master}")
print(f"Celeb Test Data successfully saved to: {save_path_test}")

--- SAVING DATAFRAME ---
Master Data successfully saved to: master_dataset.csv
Celeb Test Data successfully saved to: celeb_test_dataset.csv


In [ ]:
!zip -rq deepfake_dataset.zip F++_Split_Intra FEI_Split_Intra CelebDF_Test master_dataset.csv celeb_test_dataset.csv

updating: F++_Split_Intra/ (stored 0%)
updating: F++_Split_Intra/target_based/ (stored 0%)
updating: F++_Split_Intra/target_based/test/ (stored 0%)
updating: F++_Split_Intra/target_based/test/original/ (stored 0%)
updating: F++_Split_Intra/target_based/test/original/original_493_f2_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_152_f0_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_652_f0_ssd.jpg (deflated 1%)
updating: F++_Split_Intra/target_based/test/original/original_380_f1_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_123_f3_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_885_f3_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_369_f3_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/original_037_f3_ssd.jpg (deflated 2%)
updating: F++_Split_Intra/target_based/test/original/origi